# Summary Plots Replication

This notebook replicates the `vr4mice_summary_plots` function from `summary_dj.py`.
Each panel is generated separately for better visibility and debugging.

## Setup and Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
cd "/app/"

In [ ]:
%run env.py
%run run.py connect

In [ ]:
import datajoint as dj
from vr4mice.analysis import analysis, plotting, utils, summary_dj
from vr4mice.schema.vr4mice import GuiParams
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Configuration

In [ ]:
# Set the key for the dataset to analyze
key = {"dataset": 'Hamster_2026-01-26_1'}
database = True
save_path = "/data/summary_plots"

## Load Data

In [ ]:
# Apply plotting style
analysis.style()

# Fetch data using the summary_dj.fetch_data function
df, box_df_output = summary_dj.fetch_data(key, database)

# Process data
df = df.infer_objects()
df["dataset"] = key["dataset"]
df = df[df.iti == 0].copy()

# Align head_dir to the screen
df["head_dir"] = ((df.head_dir) + 180) % 360 - 180

# Handle occlusion
if (GuiParams() & key).fetch("occlusion_type_param") == 0.0:
    df["aperture"] = 0

num_apertures = len(df.aperture.unique())
print(f"Number of apertures: {num_apertures}")

## 1. All Trials (no ITI)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
plotting.plot_session(
    df=df,
    box_df=box_df_output,
    per_aperture=False,
    per_side=True,
    ax=ax,
)
ax.set_title("All Trials (no ITI)")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 2. Left Rewarded Trials

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
plotting.plot_session(
    df=df[(df.trial_rewarded == 1) & (df.trial_left_choice == 1)],
    box_df=box_df_output,
    per_aperture=False,
    per_side=True,
    ax=ax,
)
ax.set_title("Left Rewarded Trials (no ITI)")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 3. Right Rewarded Trials

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
plotting.plot_session(
    df=df[(df.trial_rewarded == 1) & (df.trial_right_choice == 1)],
    box_df=box_df_output,
    per_aperture=False,
    per_side=True,
    ax=ax,
)
ax.set_title("Right Rewarded Trials (no ITI)")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 4. Mean J-shaped Trajectory

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
j_shaped_df = analysis.get_jshaped_trials(df).copy()
j_shaped_df = utils.create_bins(
    data=j_shaped_df, spatial_ybins=[6.75, 20, 25], label="y"
)
plotting.lineplot_flip_axis(
    data=j_shaped_df,
    x="bin_centers",
    y="x",
    hue="trial_right_choice" if num_apertures <= 2 else "aperture",
    palette=plotting.colors_choice if num_apertures <= 2 else "viridis",
    style="aperture" if num_apertures <= 2 else "trial_right_choice",
    errorbar="se",
    ax=ax,
)
ax.set_xlim([-15, 15])
ax.set_ylabel("y position (cm)")
ax.set_xlabel("x position (cm)")
ax.set_title("Mean J-shaped Trajectory")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 5. Choice Rate

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_rate(
    df=df,
    label_x="trial_left_choice",
    per_aperture=True if num_apertures >= 2 else False,
    ax=ax,
)
ax.set_ylabel("Prob(Left Choice)")
ax.set_ylim([0, 1])
ax.hlines(
    0.5,
    xmin=ax.get_xlim()[0],
    xmax=ax.get_xlim()[1],
    colors="gray",
    linestyles="dashed",
)
ax.set_title("Choice Rate")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 6. Target Location Rate

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_rate(
    df=df,
    label_x="object_on_left",
    per_aperture=True if num_apertures >= 2 else False,
    ax=ax,
)
ax.hlines(
    0.5,
    xmin=ax.get_xlim()[0],
    xmax=ax.get_xlim()[1],
    colors="gray",
    linestyles="dashed",
)
ax.set_ylabel("Prob(Target on Left)")
ax.set_ylim([0, 1])
ax.set_title("Target Location Rate")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 7. Trial Count

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_trial_count(
    df=df,
    per_aperture=True if num_apertures >= 2 else False,
    ax=ax,
)
ax.hlines(
    y=125,
    xmin=ax.get_xlim()[0],
    xmax=ax.get_xlim()[1],
    colors="purple",
    linestyles="dashed",
)
ax.set_title("Trial Count per Session")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 8. Reward Rate

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_rewards(
    df=df,
    per_aperture=True if num_apertures >= 2 else False,
    ax=ax,
)
ax.hlines(
    0.7,
    xmin=ax.get_xlim()[0],
    xmax=ax.get_xlim()[1],
    colors="purple",
    linestyles="dashed",
)
ax.set_title("Success Rate")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 9. Time to Reward - Per Aperture

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_time_to_reward(
    df,
    label_x="aperture",
    xticks=list(df.aperture.astype(float).sort_values().astype(str).unique()),
    ax=ax,
)
ax.set_ylabel("Trial Duration / Occl")
ax.set_title("Time to Reward per Aperture")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 10. Time to Reward - Per Reward

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_time_to_reward(
    df,
    label_x="trial_rewarded",
    xticks=["Incorrect", "Correct"],
    ax=ax,
)
ax.set_ylabel("Trial Duration / Reward")
ax.set_title("Time to Reward per Outcome")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 11. Time to Reward - Per Choice

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_time_to_reward(
    df,
    label_x="trial_right_choice",
    xticks=["Left", "Right"],
    ax=ax,
)
ax.set_ylabel("Trial Duration / Choice")
ax.set_title("Time to Reward per Choice")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 12. Prepare Interpolated Data

In [ ]:
# Interpolation on variable of interest
columns = [
    "y",
    "head_dir",
    "trial_tortuosity",
    "trial_duration",
    "x",
    "aperture",
    "velocity",
    "velocity_x",
    "velocity_y",
    "trial_traj_path_length",
    "flip_one_side",
]

interpolated_df = utils.interpolate(
    df,
    n_points=200,
    value_columns=["trial_right_choice", "trial_rewarded"] + columns,
)
interpolated_df["trial_step"] = interpolated_df.groupby(
    ["dataset", "trial"]
).trial.cumcount()
interpolated_df["trial_length"] = interpolated_df["trial_step"] / 200

print(f"Interpolated data shape: {interpolated_df.shape}")

## 13. Velocity - Per Aperture

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
sns.lineplot(
    data=interpolated_df,
    x="trial_length",
    y="velocity",
    palette=(plotting.colors_aperture[:2] if num_apertures == 2 else "viridis"),
    hue="aperture",
    errorbar="se",
    ax=ax,
)
ax.set_ylabel("Speed / Aperture")
ax.set_xlabel("Trial progression")
ax.set_title("Velocity per Aperture")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 14. Velocity - Per Reward

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
sns.lineplot(
    data=interpolated_df,
    x="trial_length",
    y="velocity",
    palette=plotting.colors_rewarded,
    hue="trial_rewarded",
    errorbar="se",
    ax=ax,
)
ax.set_ylabel("Speed / Reward")
ax.set_xlabel("Trial progression")
ax.set_title("Velocity per Reward Outcome")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 15. Velocity - Per Choice

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
sns.lineplot(
    data=interpolated_df,
    x="trial_length",
    y="velocity",
    palette=plotting.colors_choice,
    hue="trial_right_choice",
    errorbar="se",
    ax=ax,
)
ax.set_ylabel("Speed / Choice")
ax.set_xlabel("Trial progression")
ax.set_title("Velocity per Choice")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 16. Heading Direction per Choice

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
sns.lineplot(
    data=interpolated_df,
    x="trial_length",
    y="head_dir",
    hue="trial_right_choice" if num_apertures <= 2 else "aperture",
    palette=plotting.colors_choice if num_apertures <= 2 else "viridis",
    style="aperture" if num_apertures <= 2 else "trial_right_choice",
    errorbar="se",
    ax=ax,
)
ax.set_ylabel("Heading direction")
ax.set_xlabel("Trial progression")
ax.set_title("Heading Direction")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 17. J-shaped Trials Rate

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plotting.plot_rate(
    df=df,
    label_x="is_j_shaped",
    per_aperture=True if num_apertures >= 2 else False,
    ax=ax,
)
ax.set_ylabel("J-shaped trials rate")
ax.set_ylim([0, 1.1])
ax.set_title("J-shaped Trials Rate")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 18. Rolling Reward

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
plotting.plot_rolling_reward(df, ax=ax, per_aperture=True if num_apertures >=2 else False)
ax.set_title("Rolling Reward")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()

## 19. Choices by Trial

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
plotting.plot_choices_by_trial(df, ax=ax)
ax.set_title("Choices by Trial")
sns.despine(ax=ax, offset=10)
plt.tight_layout()
plt.show()